# Extended-Span Validation and Superchris Channel Check (All 5 Rats)

**Two issues found in notebook 010, addressed here:**

1. **Barat and Stella peaked at +1500ms, the edge of the tested range.** We don't know if accuracy keeps
   climbing past that point, or if 1500ms genuinely is the peak, because we never tested further out.
   This notebook extends each rat's search range to that rat's OWN safe limit, computed from that rat's
   real minimum trial-to-trial spacing (not a single number assumed to be safe for all 5), so we never
   accidentally let a window run into the next trial's data.

2. **Superchris's raw RMS amplitude (0.269) was nearly double every other rat's, with far higher
   channel-to-channel variability (0.093 vs. 0.016-0.046 elsewhere).** Before crediting Superchris's
   strong decoding performance to "cleaner signal," this notebook checks whether that's true across all
   22 channels, or whether one unusually loud (possibly noisy) channel is inflating the average.

**Runtime note:** the extended range means MORE window positions are tested per rat than notebook 010,
this will take longer, correctness is prioritized here over speed, so let it run.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.preprocessing import build_labels, get_sampling_rate


## 1. Compute Each Rat's Real Minimum Trial-to-Trial Gap

Using real timestamps (not an assumed sampling rate), so this is accurate regardless of the
non-uniform-sampling issue found in notebook 007.


In [ ]:
raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])

min_gaps_ms = {}
for session_dir in session_dirs:
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    timebin = bvr_data[bvr_keys.index('TimeBin')]
    labels = build_labels(bvr_data, bvr_keys)

    trial_times = timebin[labels['trial_idx']]
    gaps_ms = np.diff(trial_times) * 1000
    min_gaps_ms[session_name] = gaps_ms.min()
    print(f"{session_name:20s} minimum trial gap: {gaps_ms.min():.0f}ms")


## 2. Set Each Rat's Safe Search Span

**Safe max offset = that rat's own minimum trial gap, minus the window length, minus a 100ms safety
buffer.** This guarantees no window can ever reach into the next trial's data, for any rat, using that
rat's actual spacing rather than a value borrowed from a different rat. Capped at a global maximum
(3000ms) purely to keep total runtime reasonable, since going further rarely makes physiological sense
for this task anyway (trials average about 25 seconds apart, but the informative window is expected to
be a small fraction of that).


In [ ]:
WINDOW_LENGTH_MS = 250
STEP_MS = 50
SPAN_START_MS = -500
SAFETY_BUFFER_MS = 100
GLOBAL_MAX_SPAN_MS = 3000
N_REPEATS = 5

BANDS = {
    'delta': (1, 4),
    'theta': (4, 12),
    'beta': (12, 30),
    'low_gamma': (30, 80),
    'high_gamma': (80, 150),
}

session_spans = {}
for session_name, min_gap in min_gaps_ms.items():
    safe_max = min_gap - WINDOW_LENGTH_MS - SAFETY_BUFFER_MS
    span_end = min(safe_max, GLOBAL_MAX_SPAN_MS)
    session_spans[session_name] = np.arange(SPAN_START_MS, span_end + 1, STEP_MS)
    print(f"{session_name:20s} span: {SPAN_START_MS}ms to {span_end:.0f}ms ({len(session_spans[session_name])} positions)")

print(f"\n--- Safety buffer used: {SAFETY_BUFFER_MS}ms ---")
print(f"(subtracted from each rat's own minimum trial gap, on top of the {WINDOW_LENGTH_MS}ms window")
print(f"length itself, before setting that rat's tested span, so windows can never reach into the")
print(f"next trial's data even in the closest-together case for that rat)")


## 3. Helper Functions (Unchanged from Notebooks 008-010)

In [ ]:
def extract_window_at_offset(lfp_data, timebin, poke_idx, offset_ms, window_ms, target_samples):
    poke_time = timebin[poke_idx]
    start_time = poke_time + offset_ms / 1000
    end_time = start_time + window_ms / 1000

    start_idx = np.searchsorted(timebin, start_time)
    end_idx = np.searchsorted(timebin, end_time)
    if end_idx - start_idx < 2:
        return None

    raw_window = lfp_data[:, start_idx:end_idx]
    n_channels, n_raw = raw_window.shape
    old_x = np.linspace(0, 1, n_raw)
    new_x = np.linspace(0, 1, target_samples)
    resampled = np.zeros((n_channels, target_samples))
    for ch in range(n_channels):
        resampled[ch] = np.interp(new_x, old_x, raw_window[ch])
    return resampled


def band_power_features(windows, fs, bands):
    n_trials, n_channels, n_samples = windows.shape
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    band_masks = {name: (freqs >= lo) & (freqs < hi) for name, (lo, hi) in bands.items()}
    n_features = n_channels * len(bands)
    X = np.zeros((n_trials, n_features))
    col = 0
    for ch in range(n_channels):
        fft_vals = np.fft.rfft(windows[:, ch, :], axis=1)
        power = np.abs(fft_vals) ** 2
        for band_name, mask in band_masks.items():
            X[:, col] = power[:, mask].sum(axis=1)
            col += 1
    return X


## 4. Extended-Span Repeated-CV Scan, Per Rat's Own Span


In [ ]:
def run_repeated_cv_for_session(session_dir, offsets_ms, window_length_ms, bands, n_repeats):
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_data = lfp['data']

    timebin = bvr_data[bvr_keys.index('TimeBin')]
    avg_fs = get_sampling_rate(bvr_data, bvr_keys)
    labels = build_labels(bvr_data, bvr_keys)
    target_samples = int(round(window_length_ms / 1000 * avg_fs))

    min_before = abs(offsets_ms[0]) / 1000
    max_after = (offsets_ms[-1] + window_length_ms) / 1000

    valid_trial_idx = []
    for t in labels['trial_idx']:
        pt = timebin[t]
        if pt - min_before < timebin[0] or pt + max_after > timebin[-1]:
            continue
        valid_trial_idx.append(t)
    valid_trial_idx = np.array(valid_trial_idx)
    valid_mask = np.isin(labels['trial_idx'], valid_trial_idx)
    inseq_outseq = labels['inseq_outseq'][valid_mask]
    odor_id = labels['odor_id'][valid_mask]

    inseq_mean, inseq_std, odor_mean, odor_std = [], [], [], []
    rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=42)

    for offset in offsets_ms:
        windows = []
        for t in valid_trial_idx:
            w = extract_window_at_offset(lfp_data, timebin, t, offset, window_length_ms, target_samples)
            windows.append(w)
        windows = np.stack(windows, axis=0)
        X = np.log1p(band_power_features(windows, avg_fs, bands))

        pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
        inseq_scores = cross_val_score(pipe, X, inseq_outseq, cv=rcv, scoring='balanced_accuracy')
        inseq_mean.append(inseq_scores.mean())
        inseq_std.append(inseq_scores.std())

        inseq_mask_local = (inseq_outseq == 1)
        X_odor = X[inseq_mask_local]
        y_odor = odor_id[inseq_mask_local]
        rcv_odor = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=42)
        odor_scores = cross_val_score(pipe, X_odor, y_odor, cv=rcv_odor, scoring='balanced_accuracy')
        odor_mean.append(odor_scores.mean())
        odor_std.append(odor_scores.std())

    return {
        'session': session_name,
        'n_trials': len(valid_trial_idx),
        'offsets_ms': offsets_ms,
        'inseq_mean': np.array(inseq_mean),
        'inseq_std': np.array(inseq_std),
        'odor_mean': np.array(odor_mean),
        'odor_std': np.array(odor_std),
    }


all_results = {}
for session_dir in session_dirs:
    session_name = session_dir.name
    print(f"Running {session_name} (span: {session_spans[session_name][0]} to {session_spans[session_name][-1]}ms)...")
    result = run_repeated_cv_for_session(session_dir, session_spans[session_name], WINDOW_LENGTH_MS, BANDS, N_REPEATS)
    all_results[session_name] = result
    print(f"  done, InSeq/OutSeq best (mean)={result['inseq_mean'].max():.3f}, "
          f"Odor best (mean)={result['odor_mean'].max():.3f}")

print("\nAll rats complete.")


## 5. Faceted Plots, Each Rat's Own Extended Span

Each panel now covers that rat's own safe range, so Barat and Stella's panels extend further right than
Superchris's (which has the tightest trial spacing and thus the smallest safe span).


In [ ]:
session_names = list(all_results.keys())
n_rats = len(session_names)

fig, axes = plt.subplots(n_rats, 2, figsize=(13, 3 * n_rats))

for row, session_name in enumerate(session_names):
    r = all_results[session_name]
    offs = r['offsets_ms']

    ax = axes[row, 0]
    ax.plot(offs, r['inseq_mean'], color='#4C72B0')
    ax.fill_between(offs, r['inseq_mean'] - r['inseq_std'], r['inseq_mean'] + r['inseq_std'], color='#4C72B0', alpha=0.25)
    ax.axhline(0.5, color='black', linestyle='--', alpha=0.4)
    ax.axvline(0, color='black', linestyle=':', alpha=0.5)
    ax.set_ylabel(session_name, fontsize=9)
    if row == 0:
        ax.set_title('InSeq/OutSeq (mean +/- 1 std)')

    ax = axes[row, 1]
    ax.plot(offs, r['odor_mean'], color='#DD8452')
    ax.fill_between(offs, r['odor_mean'] - r['odor_std'], r['odor_mean'] + r['odor_std'], color='#DD8452', alpha=0.25)
    ax.axhline(0.2, color='black', linestyle='--', alpha=0.4)
    ax.axvline(0, color='black', linestyle=':', alpha=0.5)
    if row == 0:
        ax.set_title('Odor Identity (mean +/- 1 std)')

for ax in axes[-1, :]:
    ax.set_xlabel('Offset from Poke-In (ms)')
plt.tight_layout()
plt.show()

print(f"\nSpans shown above, per rat (safety buffer: {SAFETY_BUFFER_MS}ms subtracted from each rat's")
print(f"own minimum trial gap, minus the {WINDOW_LENGTH_MS}ms window length itself):")
for session_name in session_names:
    offs = all_results[session_name]['offsets_ms']
    print(f"  {session_name:20s} tested {offs[0]:.0f}ms to {offs[-1]:.0f}ms "
          f"(that rat's min trial gap: {min_gaps_ms[session_name]:.0f}ms)")


## 6. Did the Boundary Peaks Resolve?

Checking specifically: for Barat and Stella, is the best offset now strictly INSIDE the tested range
(not sitting at the very last position), which would confirm we've found a true peak rather than an
artificially truncated one.


In [ ]:
print(f"{'Rat':20s} {'Best InSeq':12s} {'@ offset':10s} {'Span end':10s} {'At boundary?':14s}")
print("-" * 70)

extended_summary = []
for session_name in session_names:
    r = all_results[session_name]
    best_idx = r['inseq_mean'].argmax()
    best_offset = r['offsets_ms'][best_idx]
    span_end = r['offsets_ms'][-1]
    at_boundary = (best_offset == span_end)

    print(f"{session_name:20s} {r['inseq_mean'][best_idx]:<12.3f} {best_offset:<10.0f} {span_end:<10.0f} {str(at_boundary):14s}")

    odor_best_idx = r['odor_mean'].argmax()
    extended_summary.append({
        'session': session_name,
        'inseq_best_acc': round(float(r['inseq_mean'][best_idx]), 4),
        'inseq_best_offset_ms': float(best_offset),
        'inseq_at_span_boundary': bool(at_boundary),
        'span_tested_ms': [int(r['offsets_ms'][0]), int(span_end)],
        'odor_best_acc': round(float(r['odor_mean'][odor_best_idx]), 4),
        'odor_best_offset_ms': float(r['offsets_ms'][odor_best_idx]),
    })


## 7. Superchris Channel-Level Amplitude Check

Is Superchris's high average RMS amplitude spread across all 22 channels, or driven by one or a few
outlier channels? A per-channel breakdown answers this directly.


In [ ]:
superchris_dir = [p for p in session_dirs if 'superchris' in p.name][0]
lfp = np.load(superchris_dir / f'{superchris_dir.name}_lfp.npz', allow_pickle=True)
lfp_data = lfp['data']
lfp_keys = lfp['keys'].tolist()

rms_per_channel = np.sqrt(np.mean(lfp_data ** 2, axis=1))

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ['#C44E52' if v > rms_per_channel.mean() + 2 * rms_per_channel.std() else '#4C72B0' for v in rms_per_channel]
ax.bar(range(len(rms_per_channel)), rms_per_channel, color=colors)
ax.axhline(rms_per_channel.mean(), color='black', linestyle='--', label=f'mean ({rms_per_channel.mean():.3f})')
ax.set_xticks(range(len(lfp_keys)))
ax.set_xticklabels(lfp_keys, rotation=90, fontsize=7)
ax.set_ylabel('RMS amplitude')
ax.set_title('Superchris: RMS amplitude per channel (red = more than 2 std above mean)')
ax.legend()
plt.tight_layout()
plt.show()

n_outlier_channels = sum(1 for v in rms_per_channel if v > rms_per_channel.mean() + 2 * rms_per_channel.std())
print(f"\nChannels more than 2 std above the mean: {n_outlier_channels} of {len(rms_per_channel)}")
print(f"Min channel RMS: {rms_per_channel.min():.4f}, Max channel RMS: {rms_per_channel.max():.4f}, "
      f"ratio: {rms_per_channel.max() / rms_per_channel.min():.2f}x")


## 7b. Visual Check: Does Superchris's Signal Look Like Real Oscillation or Artifact?

The per-channel amplitude check above tells us WHERE the extra amplitude is, but not WHY. A quick visual
comparison helps distinguish two very different explanations: genuinely larger-amplitude but still clean
oscillatory brain signal (fine, even a good sign), versus contamination from movement or muscle artifact
(which tends to look spiky, saturated, or irregular rather than smoothly rhythmic). We plot the same
1-second raw snippet, same channel index, for Superchris and Mitt side by side, on the SAME y-axis scale
so the amplitude difference is directly visible, not just implied by a number.

Note: trial count differences between rats (Superchris 240 vs. Mitt 292) are NOT a plausible explanation
for the RMS amplitude difference seen earlier, that calculation uses the entire ~2 hour continuous
recording (millions of samples per channel), not the trial windows, so it's a very statistically stable
estimate regardless of how many trials occurred. If Superchris's amplitude is elevated, it reflects
something about the recording itself (hardware, electrode, or artifact), not sample size.


In [ ]:
mitt_dir = [p for p in session_dirs if 'mitt' in p.name][0]
lfp_mitt = np.load(mitt_dir / f'{mitt_dir.name}_lfp.npz', allow_pickle=True)
lfp_mitt_data = lfp_mitt['data']

# use the same channel index and a comparable snippet length (1 second) for both rats
channel_idx = 0
snippet_samples = 1000  # roughly 1 second at either rat's sampling rate, close enough for a visual check

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharey=True)
axes[0].plot(lfp_data[channel_idx, :snippet_samples], color='#C44E52')
axes[0].set_title(f'Superchris, channel {lfp_keys[channel_idx]}, first {snippet_samples} samples')
axes[0].set_ylabel('Voltage')

axes[1].plot(lfp_mitt_data[channel_idx, :snippet_samples], color='#4C72B0')
axes[1].set_title(f'Mitt, channel {channel_idx}, first {snippet_samples} samples (same y-axis scale)')
axes[1].set_ylabel('Voltage')
axes[1].set_xlabel('Sample')

plt.tight_layout()
plt.show()

print("What to look for: smooth, rhythmic oscillation at a similar SHAPE but larger scale in Superchris")
print("suggests genuinely larger-amplitude clean signal. Spiky, irregular, or flat-topped (saturated)")
print("segments in Superchris that don't look like the rest of the trace suggest artifact contamination.")


## 8. Text-Only Results Export


In [ ]:
import json as _json
import os

results_summary = {
    "purpose": "Extended per-rat safe-span validation, plus Superchris per-channel amplitude check",
    "window_length_ms": WINDOW_LENGTH_MS,
    "step_ms": STEP_MS,
    "safety_buffer_ms": SAFETY_BUFFER_MS,
    "min_trial_gaps_ms": {k: round(float(v), 1) for k, v in min_gaps_ms.items()},
    "session_spans_tested_ms": {k: [int(v[0]), int(v[-1])] for k, v in session_spans.items()},
    "extended_per_rat_summary": extended_summary,
    "superchris_channel_check": {
        "n_channels": len(rms_per_channel),
        "min_rms": round(float(rms_per_channel.min()), 4),
        "max_rms": round(float(rms_per_channel.max()), 4),
        "max_to_min_ratio": round(float(rms_per_channel.max() / rms_per_channel.min()), 2),
        "n_outlier_channels_2std": int(n_outlier_channels),
        "per_channel_rms": {lfp_keys[i]: round(float(rms_per_channel[i]), 4) for i in range(len(lfp_keys))},
    },
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook011_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook011_results.json")


## 9. Written Summary Report

**Objective**

Resolve two open questions from notebook 010: (1) whether Barat and Stella's InSeq/OutSeq peaks at
+1500ms were true peaks or artifacts of stopping the search too early, and (2) whether Superchris's
notably higher raw signal amplitude is spread across all channels or driven by a small number of
outliers, and whether it could simply be explained by Superchris having fewer trials than other rats.

**Method**

Each rat's search span was extended to that rat's OWN safe maximum: that rat's real minimum
trial-to-trial gap (measured directly from timestamps, not assumed), minus the 250ms window length,
minus an explicit **100ms safety buffer**. This guarantees no tested window can ever reach into the
next trial's data, for any rat, using that rat's own actual spacing rather than a number borrowed from
a different rat (exact per-rat spans and the buffer are printed directly under Section 2 and again
under the Section 5 plots). Repeated cross-validation (25 scores per window position) was re-run across
each rat's full extended span. Superchris's raw LFP was broken down into per-channel RMS amplitude, and
a 1-second raw trace was visually compared against Mitt's, on the same voltage scale, to check whether
the amplitude difference looks like clean signal or artifact contamination.

**Results**

*(Fill in after running Sections 6, 7, and 7b.)* For Barat and Stella: is the best InSeq/OutSeq offset
now strictly inside the tested range, or still at the boundary? For Superchris: how many channels exceed
2 standard deviations above the mean, what's the max-to-min channel ratio, and does the raw trace look
like clean oscillation or artifact?

**Interpretation**

*(Fill in after review.)* On the trial-count question specifically: Superchris's RMS amplitude is
computed from the full continuous recording (millions of samples per channel), not the trial windows,
so the modest difference in trial count between rats (Superchris 240 vs. Mitt 292) is not a plausible
explanation for the amplitude difference, that estimate is far too statistically stable to be
meaningfully affected by a difference of this size. The real explanation is either an outlier channel
(Section 7), or a genuine hardware/electrode/artifact difference (Section 7b), not sample size.

If Barat/Stella's peaks are now interior (not at the boundary), the extended search resolved the open
question and these can be reported as genuine peaks. If still at the boundary, that rat's true peak may
lie beyond what's physiologically reasonable to test, and the reported figures should be treated as a
lower bound, not a confirmed peak.

**Next Steps**

1. Adopt these validated, extended-span numbers as the final reference results for the Sep 1-2
   presentation.
2. If Superchris shows a clear outlier channel or artifact-like trace, consider re-running its decoding
   analysis with that channel excluded, to check whether performance holds up without it.
3. Report the odor identity finding as "broadly decodable across a wide window, without one sharp
   temporal peak" given the offset instability observed across notebooks 009 and 010.
4. Proceed to the multi-rat pooled training pipeline using the validated windows found here, with a
   session-based train/test split.
5. Run a permutation test on the strongest validated result before finalizing presentation numbers.
